In [1]:
import json
import pandas as pd
from typing import List, Dict, Any
from pydantic import BaseModel

In [2]:
class RefactoringAction(BaseModel):
    name: str = ""
    target: str = ""
    to: str = ""
    where: str = ""
    taggedValue: str
    scalingFactor: str

    def encode(self) -> str:
        if self.name.lower() == 'clone':
            return f"{self.name}___{self.target}__Cloned-Node_{self.to}__{self.where}"
        return f"{self.name}___{self.target}__{self.to}__{self.where}"

class Solution(BaseModel):
    objectives: Dict[str, Any]
    consideredObjectives: Dict[str, Any]
    refactoring: List[RefactoringAction]
    solID: int

    def encode(self) -> List[str]:
      return {
        "solID": self.solID,
        "actions": [action.encode() for action in self.refactoring],
        "objectives": self.consideredObjectives
      }

class Population(BaseModel):
    solutions: List[Solution]
    independentRun: int
    iteration: int

    def to_dataframe(self, n_actions=4) -> pd.DataFrame:
        data = [sol.encode() for sol in self.solutions]
        # Flatten refactoring actions for each solution
        # for entry in data:
        #   actions = entry.pop("actions")
        #   for i in range(n_actions):
        #       if i < len(actions):
        #           entry[f"action_{i+1}"] = actions[i]
        #       else:
        #           entry[f"action_{i+1}"] = None
        # Flatten objectives dictionary for each solution
        for entry in data:
          objectives = entry.pop("objectives")
          for obj_key, obj_value in objectives.items():
              entry[obj_key] = obj_value
        return pd.DataFrame(data)

    def get_vocabulary(self) -> List[str]:
        vocabulary = set()
        for sol in self.solutions:
            for action in sol.refactoring:
                vocabulary.add(action.encode())
        return list(vocabulary)

class Experiment(BaseModel):
    population: List[Solution]
    pareto: List[Population]
    superPareto: List[Population]
    # For now, we are skipping the configuration part
    configuration: Dict[str, Any]


In [3]:
JSON_FILENAME_EXP = "./easier_pop_json/experiment.json"

with open(JSON_FILENAME_EXP, 'r') as json_file:
    experiment_dict = json.load(json_file)

# experiment_dict
experiment = Experiment(**experiment_dict)

In [4]:
# JSON_FILENAME = "./easier_pop_json/population__1.json"
JSON_FILENAME = "./population__trainticket.json"

# JSON_FILENAME = "./easier_pop_json/population__5.json"
# JSON_FILENAME = "./population__1.json"

with open(JSON_FILENAME, 'r') as json_file:
    population_dict = json.load(json_file)

population_dict
population1 = Population(**population_dict)

In [5]:
df1 = population1.to_dataframe()
print(df1.shape)
df1

(12, 13)


,solID,actions,perfq,pricePerScenario__Login_job_class,pricePerScenario__Rebook a ticket_job_class,pas,price,changes,energyPerScenario__Login_job_class,power,energyPerScenario__Update user details_job_class,pricePerScenario__Update user details_job_class,energyPerScenario__Rebook a ticket_job_class
0,10,[mcnn___Seat__New-Node_d2.2xlarge_mbayzifiad__...,-0.152533,0.367099,0.159801,3.0,2.0219,7.70,28.869591,57.227810,13.815462,0.035700,14.542754
1,8,"[moc___login__Rebook__, moc___generate__Travel...",-0.216523,0.058770,0.252258,3.0,1.5270,8.52,7.892466,52.062848,13.793347,0.035672,30.377018
2,7,"[moc___generate__Order-other__, mcnn___Route-p...",-0.184305,0.742800,0.279300,2.0,2.6693,10.29,47.760550,95.703545,13.815462,0.035700,34.127530
3,6,"[moc___login__Rebook__, clone___continer-route...",-0.252458,0.492612,0.249888,2.0,2.4257,11.30,29.158251,72.845515,13.615482,0.035700,30.071774
4,3,[mcnn___Rebook__New-Node_t2.medium_ujgaxmcfun_...,-0.096841,0.495200,0.279300,6.0,1.8063,11.52,38.790563,79.305058,6.386996,0.004000,34.127530
5,2,[moncnn___login__New-Component_ukdqildbzu__New...,-0.258605,0.495200,0.279300,3.0,2.2381,15.12,38.794164,100.430531,27.509221,0.279300,34.127163
6,0,[mcnn___Travel-plan__New-Node_m6i.xlarge_svzxd...,-0.191981,0.495200,0.495200,2.0,2.1213,13.48,38.794164,93.397353,15.733355,0.039700,38.869870
7,30,[mcnn___Seat__New-Node_d2.2xlarge_mbayzifiad__...,-0.152533,0.367099,0.159801,3.0,2.0219,7.70,28.869591,57.227810,13.815462,0.035700,14.542754
8,27,"[moc___modify__Station__, clone___container-lo...",-0.191981,0.495200,0.495200,2.0,2.0222,11.52,38.794164,95.732286,18.068288,0.067400,38.869870
9,50,"[moc___modify__Station__, clone___container-lo...",-0.191981,0.495200,0.495200,2.0,2.0222,11.52,38.794164,95.732286,18.068288,0.067400,38.869870


In [6]:
vocab = population1.get_vocabulary()
print(len(vocab))
vocab

29


['mcnn___Verification__New-Node_d2.2xlarge_gtxyosngdu__',
 'moc___login__Rebook__',
 'moc___modify__Station__',
 'mcnn___Seat__New-Node_d2.2xlarge_mbayzifiad__',
 'clone___container-ticket-info_t2.medium__Cloned-Node_container-ticket-info_t2.medium_fyxzifsnie__',
 'moc___verify__Sso__',
 'clone___container-station_t2.medium__Cloned-Node_container-station_t2.medium_otdoiebnkv__',
 'mcnn___Route-plan__New-Node_d2.2xlarge_vckkinxguh__',
 'moc___modify__Order-other__',
 'mcnn___Ticket-info__New-Node_d2.2xlarge_embbnmwwhy__',
 'moc___login__Admin-user__',
 'moc___getByID__Login__',
 'clone___container-login_m5ad.xlarge__Cloned-Node_container-login_m5ad.xlarge_ocobkryzvi__',
 'moc___generate__Order-other__',
 'moc___updateUser__Sso__',
 'mcnn___Rebook__New-Node_t2.medium_ujgaxmcfun__',
 'clone___container-seat_t2.medium__Cloned-Node_container-seat_t2.medium_frjaaqusqy__',
 'moncnn___modify__New-Component_aulasonxla__New-Node_m5ad.xlarge_pioqpbsnan',
 'mcnn___Rebook__New-Node_m5ad.xlarge_ynyt

In [7]:
# encoding_ga_refactor.py
from __future__ import annotations
import json
import itertools
import re
from pathlib import Path
from typing import Dict, List, Union, Optional

import pandas as pd


ACTIONS = ["mcnn", "moc", "moncnn", "clone"]  # fixed vocabulary of actions
BIGRAMS = [f"{a1}_{a2}" for a1, a2 in itertools.product(ACTIONS, repeat=2)]

# --- Node family/size vocab (fixed at time zero) ---
FAMILY_PREFIXES = {"t2", "t3", "m5", "m5ad", "m6i", "c5", "c6i", "r5", "r6i", "d2"}
SIZE_CLASSES = {
    "nano", "micro", "small", "medium", "large",
    "xlarge", "2xlarge", "4xlarge", "8xlarge",
    "12xlarge", "16xlarge", "24xlarge", "32xlarge"
}
FAM_OTHER = "OTHER"
SIZE_OTHER = "OTHER"

# Matches pieces like 'm6i.xlarge', 'd2.2xlarge' anywhere in a token
_family_size_re = re.compile(
    r"([a-z]+\d+[a-z]*)\.(\d*x?large|nano|micro|small|medium|large)", re.I
)


def _pos_feature_names(max_len: int = 4) -> List[str]:
    return [f"pos{p}_{a}" for p in range(1, max_len + 1) for a in ACTIONS]


def _all_family_prefixes() -> List[str]:
    return sorted(FAMILY_PREFIXES | {FAM_OTHER})


def _all_size_classes() -> List[str]:
    return sorted(SIZE_CLASSES | {SIZE_OTHER})


def _node_feature_names() -> List[str]:
    fams = _all_family_prefixes()
    sizes = _all_size_classes()
    names: List[str] = []
    # global family counts
    names += [f"count_family_{fam}" for fam in fams]
    # per-action-to-family counts (only actions that place on a node)
    for a in ("mcnn", "moncnn", "clone"):
        names += [f"count_{a}_to_family_{fam}" for fam in fams]
    # size counts
    names += [f"count_size_{s}" for s in sizes]
    # diversity
    names += ["uniq_families_used", "uniq_sizes_used"]
    return names


def extract_family_size(token: str) -> tuple[str, str]:
    """
    Find '<family>.<size>' in a token, normalize into fixed vocab with OTHER fallback.
    Examples: 'm6i.xlarge', 'd2.2xlarge', possibly inside 'New-Node_m6i.xlarge_xyz'
    """
    if not token:
        return (FAM_OTHER, SIZE_OTHER)
    m = _family_size_re.search(token)
    if not m:
        # try substrings split by '_' if embedded
        for part in token.split("_"):
            m = _family_size_re.search(part)
            if m:
                break
    if m:
        fam = m.group(1).lower()
        size = m.group(2).lower()
        fam_norm = fam if fam in FAMILY_PREFIXES else FAM_OTHER
        size_norm = size if size in SIZE_CLASSES else SIZE_OTHER
        return (fam_norm, size_norm)
    return (FAM_OTHER, SIZE_OTHER)


def add_node_family_features(encoding: Dict, refactoring_steps: List[Dict]) -> None:
    """
    Mutates `encoding` in place, adding:
      - count_family_{fam}, count_size_{size}
      - count_{action}_to_family_{fam} for actions that place on a node (mcnn/moncnn/clone)
      - uniq_families_used, uniq_sizes_used
    """
    # Initialize all node-related columns for stability
    for fam in _all_family_prefixes():
        encoding[f"count_family_{fam}"] = 0
        encoding[f"count_mcnn_to_family_{fam}"] = 0
        encoding[f"count_moncnn_to_family_{fam}"] = 0
        encoding[f"count_clone_to_family_{fam}"] = 0
    for size in _all_size_classes():
        encoding[f"count_size_{size}"] = 0
    encoding["uniq_families_used"] = 0
    encoding["uniq_sizes_used"] = 0

    fam_seen, size_seen = set(), set()

    for step in refactoring_steps:
        name = (step.get("name") or "").strip()
        # Determine where the node info lives
        node_token = ""
        if name == "mcnn":
            node_token = step.get("to", "")
        elif name == "moncnn":
            node_token = step.get("where", "")
        elif name == "clone":
            node_token = step.get("to", "")
        else:
            # moc: no new node destination; skip
            continue

        fam, size = extract_family_size(node_token)

        # global counts
        encoding[f"count_family_{fam}"] += 1
        encoding[f"count_size_{size}"] += 1
        fam_seen.add(fam)
        size_seen.add(size)

        # per-action cross
        if name in ("mcnn", "moncnn", "clone"):
            encoding[f"count_{name}_to_family_{fam}"] += 1

    encoding["uniq_families_used"] = len(fam_seen)
    encoding["uniq_sizes_used"] = len(size_seen)


def encode_solution(
    solution: Dict,
    include_bigrams: bool = True,
    include_positional: bool = True,
    positional_len: int = 4,
) -> Dict:
    """
    Encode a single solution into a fixed-size vector.

    Features:
      - count_*: unigram counts over ACTIONS
      - bigram_*: pair counts over ACTIONS x ACTIONS (if include_bigrams=True)
      - pos{k}_{action}: one-hot for action at position k (1..positional_len) (if include_positional=True)
      - num_new_nodes_created, num_new_components_created: derived from action semantics
      - node family/size features parsed from destination/where tokens
      - sequence_len, solID
      - y__<objective>: considered objectives as targets/labels (prefixed)

    Assumptions:
      - mcnn -> creates a new node
      - moncnn -> creates a new component + new node
      - clone -> creates a new node
      - moc -> moves operation to existing component (no new elements)
    """
    ref = solution.get("refactoring", []) or []
    actions = [step.get("name", "").strip() for step in ref]
    # considered = solution.get("consideredObjectives", {}) or {}
    considered = solution.get("objectives", {}) or {}
    feats: Dict[str, Union[int, float, str]] = {}

    # Base: IDs / metadata
    feats["solID"] = solution.get("solID", None)
    feats["sequence_len"] = len(actions)

    # Unigrams
    for a in ACTIONS:
        feats[f"count_{a}"] = 0
    for a in actions:
        if a in ACTIONS:
            feats[f"count_{a}"] += 1

    # Bigrams
    if include_bigrams:
        for b in BIGRAMS:
            feats[f"bigram_{b}"] = 0
        for a1, a2 in zip(actions, actions[1:]):
            if a1 in ACTIONS and a2 in ACTIONS:
                feats[f"bigram_{a1}_{a2}"] += 1

    # Positional one-hots (fixed first N positions; missing remain 0)
    if include_positional:
        for name in _pos_feature_names(positional_len):
            feats[name] = 0
        for idx, a in enumerate(actions[:positional_len], start=1):
            if a in ACTIONS:
                feats[f"pos{idx}_{a}"] = 1

    # Creation counters (derived from action semantics)
    feats["num_new_components_created"] = feats["count_moncnn"]
    feats["num_new_nodes_created"] = (
        feats["count_mcnn"] + feats["count_moncnn"] + feats["count_clone"]
    )

    # --- Node family/size features from tokens like 'm6i.xlarge' / 'd2.2xlarge' ---
    add_node_family_features(feats, ref)

    # Targets (considered objectives)
    for k, v in considered.items():
        feats[f"y__{k}"] = v

    return feats


def encode_population(
    population: Dict,
    include_bigrams: bool = True,
    include_positional: bool = True,
    positional_len: int = 4,
    save_csv: Optional[Union[str, Path]] = None,
) -> pd.DataFrame:
    """Encode all solutions in a population JSON to a DataFrame."""
    solutions = population.get("solutions", []) or []
    rows = [
        encode_solution(
            sol,
            include_bigrams=include_bigrams,
            include_positional=include_positional,
            positional_len=positional_len,
        )
        for sol in solutions
    ]
    df = pd.DataFrame(rows)

    # Order columns for readability (keep stable)
    col_order: List[str] = []
    col_order += ["solID", "sequence_len", "num_new_nodes_created", "num_new_components_created"]
    col_order += [f"count_{a}" for a in ACTIONS]
    if include_bigrams:
        col_order += [f"bigram_{b}" for b in BIGRAMS]
    if include_positional:
        col_order += _pos_feature_names(positional_len)
    # node family/size block
    col_order += _node_feature_names()

    # targets at the end
    target_cols = sorted([c for c in df.columns if c.startswith("y__")])

    # plus any columns not covered (safe fallback)
    remaining = [c for c in df.columns if c not in set(col_order + target_cols)]
    df = df[col_order + remaining + target_cols]

    if save_csv is not None:
        save_csv = Path(save_csv)
        save_csv.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(save_csv, index=False)

    return df


def load_population(src: Union[str, Path, Dict]) -> Dict:
    """Load a population JSON from a dict, string path, or raw JSON string."""
    if isinstance(src, dict):
        return src
    p = Path(src)
    if p.exists():
        return json.loads(Path(p).read_text(encoding="utf-8"))
    # if not a file, try to parse as raw JSON string
    return json.loads(str(src))


In [9]:
population = load_population(JSON_FILENAME)
population

{'solutions': [{'objectives': {'pricePerScenario__Login_job_class': 0.36709882153467804,
    'responseTimePerScenario__Login_job_class': 345.961,
    'pas': 3.0,
    'reliability': -0.3062048505627286,
    'changes': 7.699999999999999,
    'perfq': -0.15253253014739104,
    'pricePerScenario__Rebook a ticket_job_class': 0.15980117846532196,
    'price': 2.0219,
    'responseTimePerScenario__Update user details_job_class': 216.035,
    'energyPerScenario__Login_job_class': 28.86959086377246,
    'responseTimePerScenario__Rebook a ticket_job_class': 645455.0,
    'power': 57.22781043452399,
    'energyPerScenario__Update user details_job_class': 13.815462326004504,
    'pricePerScenario__Update user details_job_class': 0.03569999620500939,
    'sysRespT': 646016.996,
    'energy': 91098.39416095124,
    'energyPerScenario__Rebook a ticket_job_class': 14.542753720751536},
   'consideredObjectives': {'perfq': -0.15253253014739104,
    'pricePerScenario__Login_job_class': 0.3670988215346780

In [9]:
# population['solutions'][4]['objectives'] = {}
# population['solutions'][7]['objectives'] = {}


In [10]:
population

{'solutions': [{'objectives': {'pricePerScenario__Login_job_class': 0.36709882153467804,
    'responseTimePerScenario__Login_job_class': 345.961,
    'pas': 3.0,
    'reliability': -0.3062048505627286,
    'changes': 7.699999999999999,
    'perfq': -0.15253253014739104,
    'pricePerScenario__Rebook a ticket_job_class': 0.15980117846532196,
    'price': 2.0219,
    'responseTimePerScenario__Update user details_job_class': 216.035,
    'energyPerScenario__Login_job_class': 28.86959086377246,
    'responseTimePerScenario__Rebook a ticket_job_class': 645455.0,
    'power': 57.22781043452399,
    'energyPerScenario__Update user details_job_class': 13.815462326004504,
    'pricePerScenario__Update user details_job_class': 0.03569999620500939,
    'sysRespT': 646016.996,
    'energy': 91098.39416095124,
    'energyPerScenario__Rebook a ticket_job_class': 14.542753720751536},
   'consideredObjectives': {'perfq': -0.15253253014739104,
    'pricePerScenario__Login_job_class': 0.3670988215346780

In [11]:
population['solutions'][4]

{'objectives': {'pricePerScenario__Login_job_class': 0.4952003728157678,
  'responseTimePerScenario__Login_job_class': 263.343,
  'pas': 6.0,
  'reliability': -0.30704655431272854,
  'changes': 11.52,
  'perfq': -0.09684097282356233,
  'pricePerScenario__Rebook a ticket_job_class': 0.2793,
  'price': 1.8063000000000002,
  'responseTimePerScenario__Update user details_job_class': 344.543,
  'energyPerScenario__Login_job_class': 38.790563226290125,
  'responseTimePerScenario__Rebook a ticket_job_class': 200044.0,
  'power': 79.30505764283998,
  'energyPerScenario__Update user details_job_class': 6.386996029373614,
  'pricePerScenario__Update user details_job_class': 0.003999998114528361,
  'sysRespT': 200651.886,
  'energy': 28346.034721461634,
  'energyPerScenario__Rebook a ticket_job_class': 34.12752981284},
 'consideredObjectives': {'perfq': -0.09684097282356233,
  'pricePerScenario__Login_job_class': 0.4952003728157678,
  'pricePerScenario__Rebook a ticket_job_class': 0.2793,
  'pas'

In [12]:
df = encode_population(
    population,
    include_bigrams=True,
    include_positional=True,
    positional_len=4,
    # save_csv=args.out,
)
df

,solID,sequence_len,num_new_nodes_created,num_new_components_created,count_mcnn,count_moc,count_moncnn,count_clone,bigram_mcnn_mcnn,bigram_mcnn_moc,...,y__power,y__price,y__pricePerScenario__Login_job_class,y__pricePerScenario__Rebook a ticket_job_class,y__pricePerScenario__Update user details_job_class,y__reliability,y__responseTimePerScenario__Login_job_class,y__responseTimePerScenario__Rebook a ticket_job_class,y__responseTimePerScenario__Update user details_job_class,y__sysRespT
0,10,4,3,0,1,1,0,2,0,0,...,57.227810,2.0219,0.367099,0.159801,0.035700,-0.306205,345.961,645455.0,216.035,646016.996
1,8,4,1,0,1,3,0,0,0,0,...,52.062848,1.5270,0.058770,0.252258,0.035672,-0.433827,30107.600,175144.0,216.280,205467.880
2,7,4,3,1,2,1,1,0,0,0,...,95.703545,2.6693,0.742800,0.279300,0.035700,-0.247378,263.343,200044.0,216.035,200523.378
3,6,4,3,0,1,1,0,2,0,0,...,72.845515,2.4257,0.492612,0.249888,0.035700,-0.432699,13589.100,175024.0,230.507,188843.607
4,3,4,3,1,1,1,1,1,0,1,...,79.305058,1.8063,0.495200,0.279300,0.004000,-0.307047,263.343,200044.0,344.543,200651.886
5,2,4,3,1,1,1,1,1,0,0,...,100.430531,2.2381,0.495200,0.279300,0.279300,-0.429242,275.895,174939.0,217.832,175432.727
6,0,4,4,1,2,0,1,1,0,0,...,93.397353,2.1213,0.495200,0.495200,0.039700,-0.309618,275.895,174984.0,216.179,175476.074
7,30,4,3,0,1,1,0,2,0,0,...,57.227810,2.0219,0.367099,0.159801,0.035700,-0.306205,345.961,645455.0,216.035,646016.996
8,27,4,3,1,1,1,1,1,0,0,...,95.732286,2.0222,0.495200,0.495200,0.067400,-0.315352,275.895,174984.0,216.179,175476.074
9,50,4,3,1,1,1,1,1,0,0,...,95.732286,2.0222,0.495200,0.495200,0.067400,-0.315352,275.895,174984.0,216.179,175476.074


In [41]:
possible_objectives = [col for col in df.columns if col.startswith('y__')]
possible_objectives

['y__changes',
 'y__energy',
 'y__energyPerScenario__ProcessSale_job_class',
 'y__energyPerScenario__ReceivedOrderedProducts_job_class',
 'y__energyPerScenario__ShowDeliveryReports_job_class',
 'y__pas',
 'y__perfq',
 'y__power',
 'y__price',
 'y__pricePerScenario__ProcessSale_job_class',
 'y__pricePerScenario__ReceivedOrderedProducts_job_class',
 'y__pricePerScenario__ShowDeliveryReports_job_class',
 'y__reliability',
 'y__responseTimePerScenario__ProcessSale_job_class',
 'y__responseTimePerScenario__ReceivedOrderedProducts_job_class',
 'y__responseTimePerScenario__ShowDeliveryReports_job_class',
 'y__sysRespT']

In [35]:
list(df.columns)

['solID',
 'sequence_len',
 'num_new_nodes_created',
 'num_new_components_created',
 'count_mcnn',
 'count_moc',
 'count_moncnn',
 'count_clone',
 'bigram_mcnn_mcnn',
 'bigram_mcnn_moc',
 'bigram_mcnn_moncnn',
 'bigram_mcnn_clone',
 'bigram_moc_mcnn',
 'bigram_moc_moc',
 'bigram_moc_moncnn',
 'bigram_moc_clone',
 'bigram_moncnn_mcnn',
 'bigram_moncnn_moc',
 'bigram_moncnn_moncnn',
 'bigram_moncnn_clone',
 'bigram_clone_mcnn',
 'bigram_clone_moc',
 'bigram_clone_moncnn',
 'bigram_clone_clone',
 'pos1_mcnn',
 'pos1_moc',
 'pos1_moncnn',
 'pos1_clone',
 'pos2_mcnn',
 'pos2_moc',
 'pos2_moncnn',
 'pos2_clone',
 'pos3_mcnn',
 'pos3_moc',
 'pos3_moncnn',
 'pos3_clone',
 'pos4_mcnn',
 'pos4_moc',
 'pos4_moncnn',
 'pos4_clone',
 'count_family_OTHER',
 'count_family_c5',
 'count_family_c6i',
 'count_family_d2',
 'count_family_m5',
 'count_family_m5ad',
 'count_family_m6i',
 'count_family_r5',
 'count_family_r6i',
 'count_family_t2',
 'count_family_t3',
 'count_mcnn_to_family_OTHER',
 'count_mcn

In [36]:
df.shape

(12, 117)

In [37]:
encoding_df = df[[x for x in df.columns if 'family' in x]]
encoding_df

,count_family_OTHER,count_family_c5,count_family_c6i,count_family_d2,count_family_m5,count_family_m5ad,count_family_m6i,count_family_r5,count_family_r6i,count_family_t2,...,count_clone_to_family_c5,count_clone_to_family_c6i,count_clone_to_family_d2,count_clone_to_family_m5,count_clone_to_family_m5ad,count_clone_to_family_m6i,count_clone_to_family_r5,count_clone_to_family_r6i,count_clone_to_family_t2,count_clone_to_family_t3
0,0,0,0,0,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,3,0,0,0,0,0,0,...,0,0,2,0,0,0,0,0,0,0
2,0,0,0,2,0,0,1,0,0,1,...,0,0,2,0,0,0,0,0,0,0
3,0,0,0,0,0,0,2,0,0,1,...,0,0,0,0,0,1,0,0,0,0
4,0,0,0,1,0,0,1,0,0,2,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,1,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
6,0,0,0,0,0,1,2,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,0,2,0,0,0,0,0,0,...,0,0,2,0,0,0,0,0,0,0
8,0,0,0,1,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
9,0,0,0,1,0,1,1,0,0,0,...,0,0,1,0,1,1,0,0,0,0


In [38]:
encoding_df.to_csv("population1_encodinng_test.csv", index=False)